# 24b HF VLM Captioning

Purpose: fill `features/{vlm_feature_id}/manifest.parquet` with mechanics captions, labels, and numeric scores using an open-source Hugging Face VLM.

Runtime: L4/A100 recommended. Requires HF token for gated models if applicable. This notebook writes only VLM feature columns, then `24_vlm_mechanics_baseline.ipynb` turns them into prediction metrics.

Default model: `Qwen/Qwen2.5-VL-3B-Instruct`.


In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


os.environ.setdefault('BATTING_CODE_ROOT', '/content/drive/MyDrive/codex/batting_codex_handoff')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path(os.environ['BATTING_CODE_ROOT'])
    mydrive = Path('/content/drive/MyDrive')
    candidates = [
        fixed,
        Path('/content/drive/MyDrive/codex/batting_codex_handoff'),
        Path('/content/drive/MyDrive/batting_codex_handoff'),
        Path.cwd(),
        *Path.cwd().parents,
    ]
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            return candidate
    raise FileNotFoundError(
        'sport_pipeline repo が見つかりません。Drive mount と repo 配置を確認してください.\n'
        f'BATTING_CODE_ROOT={fixed}\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別配置の場合は notebook の最初の cell より前に次を実行してください.\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/あなたの配置/batting_codex_handoff\n'
        f'MyDrive exists={mydrive.exists()}\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()
REPO_DIR = _resolve_repo_dir()
os.environ['BATTING_CODE_ROOT'] = str(REPO_DIR)
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME
src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(f'sport_pipeline を import できません: {src_dir}')

from sport_pipeline.pipeline.run_profile import artifact_id, load_run_profile, resolve_statcast_date_range, run_id, stage_settings

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
START_DATE, END_DATE = resolve_statcast_date_range(RUN_PROFILE)
FULL_RUN_ID = run_id(RUN_PROFILE, 'full_run_id')
VLM_RUN_ID = run_id(RUN_PROFILE, 'vlm_run_id', 'vlm_mechanics_mlb_2024_2026_v2')
VLM_FEATURE_ID = artifact_id(RUN_PROFILE, 'vlm_feature_id')
VLM_SETTINGS = stage_settings(RUN_PROFILE, 'vlm_mechanics')

print('STATCAST_DATE_RANGE =', START_DATE, 'to', END_DATE)
print('FULL_RUN_ID =', FULL_RUN_ID)
print('VLM_RUN_ID =', VLM_RUN_ID)
print('VLM_FEATURE_ID =', VLM_FEATURE_ID)


In [ ]:
# Install only in Colab when you are ready to run the open-source VLM.
install_env = os.environ.get('INSTALL_HF_VLM_DEPS')
INSTALL_HF_VLM_DEPS = (
    install_env.lower() in {'1', 'true', 'yes'}
    if install_env is not None
    else bool(VLM_SETTINGS.get('install_hf_vlm_deps_default', VLM_SETTINGS.get('execute_default', False)))
)

if INSTALL_HF_VLM_DEPS:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.49.0', 'accelerate', 'qwen-vl-utils[decord]', 'decord', 'torchvision'])
else:
    print('INSTALL_HF_VLM_DEPS=False. Set True in Colab GPU runtime before captioning.')


In [ ]:
import importlib
import sport_pipeline.models.vlm as vlm_module
import sport_pipeline.models.vlm.feature_baseline as vlm_feature_baseline_module
importlib.reload(vlm_feature_baseline_module)
importlib.reload(vlm_module)
from sport_pipeline.models.vlm import build_vlm_feature_template

MAX_CLIPS = VLM_SETTINGS.get('max_clips', 500)
vlm_manifest = BASE_DIR / f'features/{VLM_FEATURE_ID}/manifest.parquet'
if not vlm_manifest.exists():
    outputs = build_vlm_feature_template(
        BASE_DIR,
        clip_run_id=FULL_RUN_ID,
        vlm_feature_id=VLM_FEATURE_ID,
        max_clips=MAX_CLIPS,
    )
    print(outputs)
else:
    print('existing template ->', vlm_manifest)


In [ ]:
# Put your token in Colab secrets or env before running, for example:
# import os
# os.environ['HF_TOKEN'] = 'hf_...'

def _env_bool(name, default):
    value = os.environ.get(name)
    return default if value is None else value.lower() in {'1', 'true', 'yes'}

run_env = os.environ.get('RUN_HF_VLM_CAPTIONING')
RUN_HF_VLM_CAPTIONING = (
    run_env.lower() in {'1', 'true', 'yes'}
    if run_env is not None
    else bool(VLM_SETTINGS.get('run_hf_captioning_default', VLM_SETTINGS.get('execute_default', False)))
)
HF_VLM_MODEL_ID = os.environ.get('HF_VLM_MODEL_ID') or VLM_SETTINGS.get('hf_model_id', 'Qwen/Qwen2.5-VL-3B-Instruct')
INPUT_MODE = os.environ.get('HF_VLM_INPUT_MODE') or VLM_SETTINGS.get('input_mode', 'clip_video')  # clip_video or debug_frame
VIDEO_READER_BACKEND = (
    os.environ.get('HF_VLM_VIDEO_READER_BACKEND')
    or os.environ.get('QWENVL_VIDEO_READER')
    or VLM_SETTINGS.get('video_reader_backend', 'decord')
)
FALLBACK_TO_DEBUG_FRAME = _env_bool('HF_VLM_FALLBACK_TO_DEBUG_FRAME', bool(VLM_SETTINGS.get('fallback_to_debug_frame', True)))
MAX_ROWS = int(os.environ.get('HF_VLM_MAX_ROWS') or VLM_SETTINGS.get('caption_max_rows', 100))
FPS = float(VLM_SETTINGS.get('caption_fps', 1.0))
MAX_PIXELS = int(VLM_SETTINGS.get('caption_max_pixels', 360 * 420))
CACHE_INPUTS = bool(VLM_SETTINGS.get('cache_inputs', False))
CACHE_MIN_FREE_DISK_GB = float(VLM_SETTINGS.get('cache_min_free_disk_gb', 20))
CACHE_MAX_FILE_MB = VLM_SETTINGS.get('cache_max_file_mb')
print('RUN_HF_VLM_CAPTIONING =', RUN_HF_VLM_CAPTIONING)
print('HF_VLM_MODEL_ID =', HF_VLM_MODEL_ID)
print('INPUT_MODE =', INPUT_MODE)
print('VIDEO_READER_BACKEND =', VIDEO_READER_BACKEND)
print('FALLBACK_TO_DEBUG_FRAME =', FALLBACK_TO_DEBUG_FRAME)
print('MAX_ROWS target complete rows =', MAX_ROWS)

from sport_pipeline.io import read_table
import importlib
import sport_pipeline.models.vlm as vlm_module
import sport_pipeline.models.vlm.hf_captioning as vlm_hf_captioning_module
importlib.reload(vlm_hf_captioning_module)
importlib.reload(vlm_module)
from sport_pipeline.models.vlm import fill_vlm_manifest_with_hf

if RUN_HF_VLM_CAPTIONING:
    outputs = fill_vlm_manifest_with_hf(
        BASE_DIR,
        vlm_feature_id=VLM_FEATURE_ID,
        model_id=HF_VLM_MODEL_ID,
        input_mode=INPUT_MODE,
        max_rows=MAX_ROWS,
        fps=FPS,
        max_pixels=MAX_PIXELS,
        video_reader_backend=VIDEO_READER_BACKEND,
        fallback_to_debug_frame=FALLBACK_TO_DEBUG_FRAME,
        cache_dir=CACHE_DIR,
        cache_inputs=CACHE_INPUTS,
        cache_min_free_disk_gb=CACHE_MIN_FREE_DISK_GB,
        cache_max_file_mb=CACHE_MAX_FILE_MB,
    )
    for name, path in outputs.items():
        print(name, '->', path, 'exists=', path.exists())
    manifest_rows = read_table(outputs['manifest']) if outputs['manifest'].exists() else []
    complete_rows = sum(1 for row in manifest_rows if row.get('vlm_caption') or row.get('feature_status') == 'vlm_complete')
    failed_rows = sum(1 for row in manifest_rows if row.get('feature_status') == 'vlm_failed')
    fallback_rows = sum(1 for row in manifest_rows if row.get('vlm_input_mode') == 'debug_frame_fallback')
    print('vlm manifest rows =', len(manifest_rows), 'complete =', complete_rows, 'failed =', failed_rows, 'debug_frame_fallback =', fallback_rows)
else:
    print('RUN_HF_VLM_CAPTIONING=False. Set True after deps/token/GPU are ready.')
    print('NEXT after captioning: run 24_vlm_mechanics_baseline.ipynb with RUN_BASELINE=True.')
